In [1]:
import sys
from pathlib import Path

import torch
from torchvision.transforms import v2 as transforms

this_path = Path(__file__) if '__file__' in globals() else Path("<undefined>.ipynb").resolve()
work_path = next((p for p in this_path.parents if p.name == "research"), None)
tools_path = work_path / Path("../torch-tools")
sys.path.append(str(tools_path))

ee_tools_path_p = work_path / Path("ee")
sys.path.append(str(ee_tools_path_p))

from network import Network
from trainer import Trainer
from datasets import fetch_handler

from torch import nn
from torch.fx import symbolic_trace

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:
from torchvision.models import resnet18 as res18_tv
# from models.resnet import resnet18 as res18_mine

from torchvision.models import mobilenet_v2 as mob_tv
# from models.mobilenet_v2_cifar import mobilenet_v2 as mine
# from ee_tools.models.mobilenet_v2_cifar_ee import mobilenet_v2 as ee
# from ee_tools.models.mobilenet_v2_ee import mobilenet_v2 as ee


In [3]:
base_network = res18_tv(num_classes=100)
network = Network(base_network)
net_name = "tv"

p = Path(this_path.parent / f"{net_name}.txt")
ti = network.torchinfo(input_size=(1, 3, 32, 32))
p.write_text(str(ti))


12637

In [4]:
# base_network = mob_tv(num_classes=100)
# network = Network(base_network)
# net_name = "tv"
#
# p = Path(this_path.parent / f"{net_name}.txt")
# ti = network.torchinfo(input_size=(1, 3, 32, 32))
# p.write_text(str(ti))
#

In [5]:
gm = symbolic_trace(base_network)
# gm.graph.print_tabular()
gm.recompile()

PythonCode(src='\n\n\ndef forward(self, x : torch.Tensor) -> torch.Tensor:\n    conv1 = self.conv1(x);  x = None\n    bn1 = self.bn1(conv1);  conv1 = None\n    relu = self.relu(bn1);  bn1 = None\n    maxpool = self.maxpool(relu);  relu = None\n    layer1_0_conv1 = getattr(self.layer1, "0").conv1(maxpool)\n    layer1_0_bn1 = getattr(self.layer1, "0").bn1(layer1_0_conv1);  layer1_0_conv1 = None\n    layer1_0_relu = getattr(self.layer1, "0").relu(layer1_0_bn1);  layer1_0_bn1 = None\n    layer1_0_conv2 = getattr(self.layer1, "0").conv2(layer1_0_relu);  layer1_0_relu = None\n    layer1_0_bn2 = getattr(self.layer1, "0").bn2(layer1_0_conv2);  layer1_0_conv2 = None\n    add = layer1_0_bn2 + maxpool;  layer1_0_bn2 = maxpool = None\n    layer1_0_relu_1 = getattr(self.layer1, "0").relu(add);  add = None\n    layer1_1_conv1 = getattr(self.layer1, "1").conv1(layer1_0_relu_1)\n    layer1_1_bn1 = getattr(self.layer1, "1").bn1(layer1_1_conv1);  layer1_1_conv1 = None\n    layer1_1_relu = getattr(self.l

In [ ]:
from ee_tools.modules.ops import RepeatData, GroupedLinear, ChunkMerge

ee_div = 2

gm = symbolic_trace(base_network)
modules_dict = dict(gm.named_modules())
# print(gm.code)
gm.add_module("repeat_data", RepeatData(ee_div ** 2))
gm.add_module("chunk_merge", ChunkMerge(ee_div ** 2))
first_input_flag = True
first_layer_flag = True

for node in list(gm.graph.nodes):
    print(node.target)
    if node.op == "placeholder" and first_input_flag:
        with gm.graph.inserting_after(node):
            new_node = gm.graph.call_module('repeat_data', args=(node,))
        node.replace_all_uses_with(new_node)
        new_node.replace_input_with(new_node, node)
        first_input_flag = False

    if node.op == 'call_module':
        module = modules_dict.get(node.target)
        # ConvNd
        if isinstance(module, (nn.Conv1d, nn.Conv2d, nn.Conv3d)):
            new_in_channels = module.in_channels * ee_div
            if first_layer_flag:
                new_in_channels *= ee_div
                first_layer_flag = False
            new_node = module.__class__(
                    in_channels=new_in_channels,
                    out_channels=module.out_channels * ee_div,
                    kernel_size=module.kernel_size,
                    stride=module.stride,
                    padding=module.padding,
                    dilation=module.dilation,
                    groups=module.groups * ee_div ** 2,
                    bias=module.bias is not None,
                    padding_mode=module.padding_mode,
                    device=module.weight.device,
                    dtype=module.weight.dtype,
                )
            setattr(gm, node.target, new_node)

        if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            new_node = module.__class__(
                    num_features=module.num_features * ee_div,
                    eps=module.eps,
                    momentum=module.momentum,
                    affine=module.affine,
                    track_running_stats=module.track_running_stats,
                    device=module.weight.device if module.affine else None,
                    dtype=module.weight.dtype if module.affine else None,
                )
            setattr(gm, node.target, new_node)

        if isinstance(module, nn.Linear):
            new_node = GroupedLinear(
                    in_features=module.in_features * ee_div,
                    out_features=module.out_features * ee_div,
                    bias=module.bias is not None,
                    groups=ee_div ** 2,
                    device=module.weight.device,
                    dtype=module.weight.dtype,
                )
            setattr(gm, node.target, new_node)
            
last_node = list(gm.graph.nodes)[-1]
with gm.graph.inserting_after(last_node):
    new_node = gm.graph.call_module('chunk_merge', args=(last_node,))
    new_node.replace_input_with(new_node, node)

gm.graph.lint()
gm.recompile()


x
conv1
bn1
relu
maxpool
layer1.0.conv1
layer1.0.bn1
layer1.0.relu
layer1.0.conv2
layer1.0.bn2
<built-in function add>
layer1.0.relu
layer1.1.conv1
layer1.1.bn1
layer1.1.relu
layer1.1.conv2
layer1.1.bn2
<built-in function add>
layer1.1.relu
layer2.0.conv1
layer2.0.bn1
layer2.0.relu
layer2.0.conv2
layer2.0.bn2
layer2.0.downsample.0
layer2.0.downsample.1
<built-in function add>
layer2.0.relu
layer2.1.conv1
layer2.1.bn1
layer2.1.relu
layer2.1.conv2
layer2.1.bn2
<built-in function add>
layer2.1.relu
layer3.0.conv1
layer3.0.bn1
layer3.0.relu
layer3.0.conv2
layer3.0.bn2
layer3.0.downsample.0
layer3.0.downsample.1
<built-in function add>
layer3.0.relu
layer3.1.conv1
layer3.1.bn1
layer3.1.relu
layer3.1.conv2
layer3.1.bn2
<built-in function add>
layer3.1.relu
layer4.0.conv1
layer4.0.bn1
layer4.0.relu
layer4.0.conv2
layer4.0.bn2
layer4.0.downsample.0
layer4.0.downsample.1
<built-in function add>
layer4.0.relu
layer4.1.conv1
layer4.1.bn1
layer4.1.relu
layer4.1.conv2
layer4.1.bn2
<built-in functio

In [7]:
network = Network(base_network)
net_name = "tv"

p = Path(this_path.parent / f"{net_name}.txt")
ti = network.torchinfo(input_size=(1, 3, 32, 32), depth=1)
p.write_text(str(ti))

print(network.count_params(trainable=False))
print(network.count_params(trainable=True))


11227812
11227812


In [8]:
network = Network(gm)
net_name = "ee"

p = Path(this_path.parent / f"{net_name}.txt")
# ti = network.torchinfo(input_size=(1, 3, 32, 32), depth=1)
p.write_text(str(ti))

print(network.count_params(trainable=False))
print(network.count_params(trainable=True))


22413896
22413896


In [9]:
input = torch.randn(4, 3, 224, 224).to(device)
t_output = base_network(input)

In [10]:
input = torch.randn(4, 3, 224, 224).to(device)
t_output = gm(input)

RuntimeError: Given groups=1, weight of size [64, 64, 3, 3], expected input[4, 128, 56, 56] to have 64 channels, but got 128 channels instead